# NextStepAI - Whisper GPU API
Chọn **Runtime > Change runtime type > T4 GPU**, sau đó chạy lần lượt tất cả cell. API tương thích với frontend hiện tại.

In [ ]:
%pip -q install faster-whisper 'av<19' fastapi uvicorn python-multipart pyngrok nvidia-cublas-cu12 'nvidia-cudnn-cu12==9.*'

In [ ]:
from pathlib import Path
server_code = r'''
import os, re, time, uuid, threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from faster_whisper import WhisperModel

app = FastAPI(title="NextStepAI Colab STT")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
jobs = {}
jobs_lock = threading.Lock()
executor = ThreadPoolExecutor(max_workers=1)
upload_dir = Path("/content/stt_uploads")
upload_dir.mkdir(exist_ok=True)
print("Loading turbo on GPU...", flush=True)
model = WhisperModel("turbo", device="cuda", compute_type="float16")
print("Whisper GPU ready", flush=True)

def update(job_id, **values):
    with jobs_lock: jobs[job_id].update(values)

def transcribe_job(job_id):
    job = jobs[job_id]
    try:
        update(job_id, status="processing", stage="transcribing", progress=35)
        started = time.perf_counter()
        segments_iter, info = model.transcribe(
            job["audio_path"], language="vi", task="transcribe", beam_size=1,
            vad_filter=True, vad_parameters={"min_silence_duration_ms": 400, "speech_pad_ms": 200},
            initial_prompt=job["context"] or None, condition_on_previous_text=False,
        )
        segments = [{"start": round(s.start,3), "end": round(s.end,3), "text": s.text.strip()} for s in segments_iter]
        raw = " ".join(s["text"] for s in segments).strip()
        inference_ms = round((time.perf_counter()-started)*1000, 3)
        result = {
            "raw_transcript": raw, "corrected_transcript": raw, "segments": segments,
            "edits": [], "needs_review": False, "language": getattr(info, "language", "vi"),
            "language_probability": getattr(info, "language_probability", None),
            "latency_ms": inference_ms, "inference_ms": inference_ms, "correction_ms": 0, "model": "turbo-gpu",
        }
        update(job_id, status="completed", stage="completed", progress=100, result=result)
    except Exception as exc:
        update(job_id, status="failed", stage="failed", progress=100, error=str(exc))

@app.get("/")
def root(): return {"status": "ok", "service": "NextStepAI Whisper GPU", "health": "/health", "create_job": "POST /api/v1/stt/jobs"}

@app.get("/health")
def health(): return {"status": "ok", "model": "turbo", "device": "cuda", "compute_type": "float16"}

@app.post("/api/v1/stt/jobs", status_code=202)
async def create_job(audio: UploadFile=File(...), interview_id: str=Form("demo"), question_id: str=Form("question"), context: str=Form("")):
    job_id = uuid.uuid4().hex
    suffix = Path(audio.filename or "answer.webm").suffix or ".webm"
    path = upload_dir / f"{job_id}{suffix}"
    path.write_bytes(await audio.read())
    job = {"id":job_id, "interview_id":interview_id, "question_id":question_id, "status":"queued", "stage":"queued", "progress":5, "result":None, "error":None, "audio_path":str(path), "context":context}
    with jobs_lock: jobs[job_id] = job
    executor.submit(transcribe_job, job_id)
    return {k:v for k,v in job.items() if k not in {"audio_path","context"}}

@app.get("/api/v1/stt/jobs/{job_id}")
def get_job(job_id: str):
    with jobs_lock: job = jobs.get(job_id)
    if not job: raise HTTPException(404, "STT job not found")
    return {k:v for k,v in job.items() if k not in {"audio_path","context"}}
'''
Path('/content/colab_stt_server.py').write_text(server_code, encoding='utf-8')
print('Server file created')

In [ ]:
import os, re, subprocess, sys, time, urllib.request
old_api = globals().get('api')
if old_api is not None and old_api.poll() is None:
    old_api.terminate()
    try: old_api.wait(timeout=10)
    except subprocess.TimeoutExpired: old_api.kill()
import nvidia.cublas.lib, nvidia.cudnn.lib
cublas_path = next(iter(nvidia.cublas.lib.__path__))
cudnn_path = next(iter(nvidia.cudnn.lib.__path__))
lib_path = cublas_path + ':' + cudnn_path
env = os.environ.copy(); env['LD_LIBRARY_PATH'] = lib_path + ':' + env.get('LD_LIBRARY_PATH','')
api_log = open('/content/stt_api.log', 'w')
api = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'colab_stt_server:app', '--host', '0.0.0.0', '--port', '8000'], cwd='/content', env=env, stdout=api_log, stderr=subprocess.STDOUT)
print('Đang tải model turbo lên GPU; lần đầu có thể mất vài phút...')
for _ in range(180):
    if api.poll() is not None:
        api_log.flush(); raise RuntimeError(open('/content/stt_api.log').read())
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2).read(); break
    except Exception: time.sleep(2)
else: raise RuntimeError('Whisper API không sẵn sàng sau 6 phút. Xem /content/stt_api.log')
print('Whisper GPU đã sẵn sàng. Đang tạo ngrok tunnel...')
from getpass import getpass
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError
ngrok.kill()
ngrok_token = getpass('Dán ngrok authtoken (ký tự sẽ được ẩn): ')
try:
    ngrok.set_auth_token(ngrok_token.strip())
except PyngrokNgrokError as exc:
    if 'Authtoken already set' not in str(exc): raise
    print('Ngrok authtoken đã được lưu từ lần chạy trước; tiếp tục tạo tunnel.')
tunnel = ngrok.connect(addr=8000, proto='http')
public_url = tunnel.public_url.replace('http://', 'https://')
for _ in range(30):
    try:
        urllib.request.urlopen(public_url + '/health', timeout=5).read(); break
    except Exception: time.sleep(1)
else: raise RuntimeError('ngrok đã tạo URL nhưng không truy cập được /health')
print('VITE_API_BASE_URL=' + public_url, flush=True)
print('Health=' + public_url + '/health', flush=True)
print('GIỮ CELL NÀY TIẾP TỤC CHẠY. Không bấm Stop.', flush=True)
while api.poll() is None: time.sleep(30)
raise RuntimeError('Whisper API đã dừng. Hãy xem /content/stt_api.log')